> **Versi publik untuk arsip repositori.** Kunci API dibaca dari Colab Secrets (`GEMINI_API_KEY`, `OPENAI_API_KEY`); tidak ada kunci di notebook ini.

# Studi C — Penilaian LLM (*Membaca Angka, Menilai Riset*)

Notebook ini menilai **versi Indonesia dan versi Inggris** setiap artikel dengan Gemini dan ChatGPT (5 ulangan per versi per model), memakai data dari notebook pertama (`translations_final.csv`, `articles.csv`) di folder Drive `StudiBuku`.

**Cara pakai**
1. Runtime biasa (**CPU**) sudah cukup — tidak perlu GPU.
2. Pastikan 🔑 *Secrets* berisi `GEMINI_API_KEY` dan `OPENAI_API_KEY` (sakelar *Notebook access* menyala).
3. Jalankan **Langkah 0–2** dulu (uji coba 4 panggilan). Bila hasil uji baik, jalankan sel fungsi Langkah 3, lalu **3a (ChatGPT)** dan **3b (Gemini)**, boleh pada waktu berbeda.
4. Bila terputus: jalankan Langkah 0–1, sel fungsi Langkah 3, lalu 3a atau 3b lagi; panggilan yang sudah tersimpan tidak diulang.

## Langkah 0 — Persiapan dan data

In [ ]:
!pip -q install google-genai openai
import os, re, json, time, random
import pandas as pd
from google.colab import drive, userdata
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/StudiBuku'

art = pd.read_csv(f'{OUT}/articles.csv', keep_default_na=False).drop_duplicates('article_id')
txt = pd.read_csv(f'{OUT}/translations_final.csv', keep_default_na=False)
txt = txt[txt.article_id.isin(art.article_id)].reset_index(drop=True)
assert len(txt) == len(art), 'translations_final.csv belum lengkap: jalankan notebook pertama sampai Langkah 3b.'
assert (txt.text_abstract_id != txt.text_abstract_en).all()
print('Artikel:', len(txt)); print(txt.bahasa_asli_v.value_counts())

## Langkah 1 — Model dan instruksi penilaian

In [ ]:
from google import genai
from openai import OpenAI
gem = genai.Client(api_key=userdata.get('GEMINI_API_KEY'), http_options={'timeout': 120000})   # batas tunggu 120 detik
oai = OpenAI(api_key=userdata.get('OPENAI_API_KEY'), timeout=120)
GEMINI_MODEL = 'gemini-3.8-flash'
OPENAI_MODEL = 'gpt-4.1-mini-2025-04-14'
REPEATS = 5

SYSTEM = (
 "You are an expert academic reviewer assessing the quality of a journal article from its title and abstract, "
 "using three criteria: originality, significance, and rigour. Give one overall score: "
 "4 = world-leading, 3 = internationally excellent, 2 = recognised internationally, 1 = recognised nationally. "
 "The title and abstract may be written in Indonesian or English; judge the research, not the language. "
 'Reply with JSON only: {"score": <1-4>, "reason": "<one sentence>"}')

def ask(model_kind, title, abstract):
    user = f"Title: {title}\n\nAbstract: {abstract}"
    err = ''
    for attempt in range(6):
        try:
            if model_kind == 'gemini':
                r = gem.models.generate_content(model=GEMINI_MODEL, contents=user,
                        config={'system_instruction': SYSTEM})
                out, ver = (r.text or ''), (getattr(r, 'model_version', None) or GEMINI_MODEL)
            else:
                r = oai.chat.completions.create(model=OPENAI_MODEL,
                        messages=[{'role':'system','content':SYSTEM}, {'role':'user','content':user}])
                out, ver = (r.choices[0].message.content or ''), r.model
            m = re.search(r'"score"\s*:\s*([1-4](?:\.\d+)?)', out)
            return (float(m.group(1)) if m else None), out, ver
        except Exception as e:
            err = repr(e)[:300]
            print(f'  [{model_kind}] percobaan {attempt+1} gagal: {err[:150]}')
            time.sleep(min(60, 5 * 2**attempt))
    return None, 'ERROR: ' + err, ''
print('Siap:', GEMINI_MODEL, '|', OPENAI_MODEL)

## Langkah 2 — Uji coba (4 panggilan, tidak disimpan)

Bila muncul `ERROR` berisi *not found* / *404*, nama model sudah berubah: jalankan sel daftar model di bawahnya, ganti `GEMINI_MODEL` atau `OPENAI_MODEL` di Langkah 1, lalu ulangi. Bila *401* / *invalid api key*, periksa Secrets. Bila *429* / *quota*, periksa billing.

In [ ]:
a = txt.iloc[0]
for mk in ('gemini', 'openai'):
    for lang in ('id', 'en'):
        score, raw, ver = ask(mk, a[f'text_title_{lang}'], a[f'text_abstract_{lang}'])
        print(f'{mk:6} | {lang} | skor={score} | versi={ver}\n   {raw[:200]}\n')

In [ ]:
# (opsional) daftar model yang tersedia untuk akun Bapak
print([m.name for m in gem.models.list() if 'gemini' in m.name][:30])
print(sorted(m.id for m in oai.models.list() if m.id.startswith('gpt'))[:40])

## Langkah 3 — Penilaian penuh

Sel pertama menyiapkan fungsi. **Sel 3a menjalankan ChatGPT** (±1 jam), **sel 3b menjalankan Gemini** (±4 jam). Keduanya boleh dijalankan pada waktu berbeda, misalnya Gemini saat servernya tidak sibuk. Setiap jawaban langsung disimpan di `llm_scores.jsonl`; panggilan yang gagal tidak disimpan dan dicoba lagi saat sel dijalankan ulang.

In [ ]:
sc_path = f'{OUT}/llm_scores.jsonl'

def jalankan(models):
    done = set()
    if os.path.exists(sc_path):
        for line in open(sc_path):
            d = json.loads(line); done.add((d['article_id'], d['lang'], d['model'], d['rep']))
    jobs = [(a, lang, mk, rep) for _, a in txt.iterrows() for lang in ('id', 'en')
            for mk in models for rep in range(REPEATS)
            if (a.article_id, lang, mk, rep) not in done]
    random.Random(42).shuffle(jobs)   # urutan acak agar bahasa tidak berurutan
    print('Model:', models, '| sudah tersimpan (semua model):', len(done), '| sisa panggilan:', len(jobs))
    gagal = 0
    with open(sc_path, 'a') as f:
        for i, (a, lang, mk, rep) in enumerate(jobs, 1):
            score, raw, ver = ask(mk, a[f'text_title_{lang}'], a[f'text_abstract_{lang}'])
            if raw.startswith('ERROR'):
                gagal += 1; print('Gagal (dicoba lagi saat dijalankan ulang):', raw[:120])
                if gagal >= 20: print('Terlalu banyak gagal berturut-turut; berhenti. Coba lagi nanti.'); break
                continue
            gagal = 0
            f.write(json.dumps({'article_id':a.article_id, 'lang':lang, 'model':mk, 'rep':rep, 'score':score,
                                'model_version':ver, 'waktu':time.strftime('%Y-%m-%d %H:%M:%S'),
                                'raw':raw[:500]}) + '\n'); f.flush()
            if i % 100 == 0: print(i, '/', len(jobs))
            time.sleep(0.5 if mk == 'openai' else 4.5)   # hormati batas laju API
    print('Selesai putaran ini.')
print('Fungsi siap.')

### Langkah 3a — ChatGPT

In [ ]:
jalankan(('openai',))

### Langkah 3b — Gemini

In [ ]:
jalankan(('gemini',))

## Langkah 4 — Ringkasan cepat

In [ ]:
sc = pd.read_json(sc_path, lines=True)
sc = sc[sc.article_id.isin(txt.article_id)]
print('Total jawaban:', len(sc), '| tanpa skor terbaca:', sc.score.isna().sum())
print('Kelengkapan (target', len(txt) * 2 * 2 * REPEATS, '):', len(sc.dropna(subset=['score'])))
s = sc.dropna(subset=['score'])
m = s.groupby(['article_id','model','lang'])['score'].mean().unstack('lang').reset_index()
m['selisih_en_minus_id'] = m['en'] - m['id']
m = m.merge(art[['article_id','field','rank']], on='article_id').merge(txt[['article_id','bahasa_asli_v']], on='article_id')
print(m.groupby(['model','field'])[['id','en','selisih_en_minus_id']].mean().round(3))
print(m.groupby(['model','bahasa_asli_v'])['selisih_en_minus_id'].agg(['mean','count']).round(3))
m.to_csv(f'{OUT}/llm_means.csv', index=False)
print('Kirim dari folder StudiBuku: llm_scores.jsonl (dan llm_means.csv)')